# Demostración: datos sintéticos para clasificación binaria desbalanceada

Este notebook ejecuta, de principio a fin y en pocos minutos, el protocolo de la tesis sobre **un dataset**:

1. Carga y partición sin fuga de información (80/20 y partición interna 80/20).
2. Generación de la clase minoritaria con **SMOTE-NC, CTGAN, TVAE y el híbrido SMOTE-NC → CTGAN**.
3. **Fidelidad** de los registros sintéticos frente a la minoría real.
4. **Utilidad**: XGBoost y CatBoost entrenados con REAL, REAL-CW (pesos de clase) y REAL + sintéticos.
5. **Validación estadística** con pruebas pareadas y corrección de Holm, y comparación con las 10 corridas publicadas.

Esquivel, Quispe, Vega y Saboya · *IJACSA* 17(9), 2026 · DOI 10.14569/IJACSA.2026.0170971

## 1. Configuración

In [ ]:
DATASET = "DS5"    # DS5: 6 819 registros, desbalance 30:1 (muestra el efecto con claridad)
                   # DS2: 383 registros (saturado: todas las condiciones superan 0.90 de Recall)
                   # DS3: 2 139 registros
MODO = "rapido"    # "rapido":   SMOTE-NC, 5 corridas (1 a 2 minutos en CPU)
                   # "completo": los 4 métodos, 3 corridas (15 a 25 minutos en CPU; el híbrido es el más lento)

if MODO == "rapido":
    N_RUNS, METODOS = 5, ["SMOTE"]
else:
    N_RUNS, METODOS = 3, ["SMOTE", "CTGAN", "TVAE", "HYBRID"]

In [ ]:
import json, math, os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, yaml
from scipy import stats
from scipy.spatial.distance import pdist
from scipy.stats import wasserstein_distance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (average_precision_score, precision_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
from imblearn.over_sampling import SMOTE, SMOTENC
from ctgan import CTGAN
from sdv.metadata import SingleTableMetadata
from sdv.single_table import TVAESynthesizer
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 4)
FOLDER = {"DS2": "dataset_02_thyroid_cancer", "DS3": "dataset_03_aids_trial_175",
          "DS5": "dataset_05_taiwanese_bankruptcy"}[DATASET]
config = yaml.safe_load(open(f"{FOLDER}/dataset_config.yaml"))
params = json.load(open(f"{FOLDER}/outputs/metrics_v2/used_configs.json"))   # hiperparámetros seleccionados
num_cols, cat_cols = config["numerical_features"], config["categorical_features"]
print(f"{DATASET}: {config['dataset_name']} | {len(num_cols)} numéricas, {len(cat_cols)} categóricas")
print("Hiperparámetros:", {k: params[k] for k in ["smote_k", "ctgan", "tvae", "hybrid", "alpha_smote"]})

## 2. Datos y partición sin fuga de información
La prueba se separa antes de cualquier transformación. Codificación y escalado se ajustan solo con entrenamiento. Los generadores y los clasificadores usan el **entrenamiento interno**.

In [ ]:
def load_uci(uci_id):
    """Lee el dataset guardado en data/; si no existe, lo descarga de la UCI y lo guarda (requiere Internet una vez)."""
    path = f"data/uci_{uci_id}.csv"
    if not os.path.exists(path):
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=uci_id)
        os.makedirs("data", exist_ok=True)
        pd.concat([d.data.features, d.data.targets], axis=1).to_csv(path, index=False)
    return pd.read_csv(path)


df = load_uci(config["uci_id"])
wanted = set(num_cols) | set(cat_cols) | {config["target_column"]}
df = df.rename(columns={c: c.strip() for c in df.columns if c not in wanted and c.strip() in wanted})
X, y = df[num_cols + cat_cols].copy(), df[config["target_column"]]

# Partición 80/20 estratificada (semilla 42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
if y_tr.dtype == object:
    le_y = LabelEncoder().fit(y_tr); y_tr, y_te = le_y.transform(y_tr), le_y.transform(y_te)
else:
    y_tr, y_te = y_tr.values, y_te.values
for c in cat_cols:
    le = LabelEncoder().fit(X_tr[c].astype(str))
    X_tr[c], X_te[c] = le.transform(X_tr[c].astype(str)), le.transform(X_te[c].astype(str))
scaler = StandardScaler().fit(X_tr[num_cols].astype(float))
def to_scaled(d):
    d = d.copy(); d[num_cols] = scaler.transform(d[num_cols].astype(float)); return d
def to_raw(d):
    d = d.copy(); d[num_cols] = scaler.inverse_transform(d[num_cols].astype(float)); return d
X_te_s = to_scaled(X_te)

# Partición interna 80/20: solo el entrenamiento interno se usa para generar y entrenar
idx_in, _ = train_test_split(np.arange(len(X_tr)), test_size=0.2, stratify=y_tr, random_state=42)
X_in_raw, y_in = X_tr.iloc[idx_in].reset_index(drop=True), y_tr[idx_in]
X_in = to_scaled(X_in_raw)
minority = int(pd.Series(y_in).value_counts().idxmin())
M, m = int((y_in != minority).sum()), int((y_in == minority).sum())
real_min = X_in_raw[y_in == minority].reset_index(drop=True)
n_gen = M - m
print(f"Entrenamiento interno: {M} mayoría / {m} minoría (ratio {M/m:.2f}:1) -> generar {n_gen} registros")
print(f"Prueba: {len(y_te)} registros, {int((y_te == minority).sum())} positivos")

## 3. Generadores (devuelven solo los registros sintéticos, en escala original)

In [ ]:
USE_GPU = torch.cuda.is_available()
cat_idx = [X_in.columns.get_loc(c) for c in cat_cols]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def smote(strategy, seed):
    if cat_idx:
        return SMOTENC(categorical_features=cat_idx, k_neighbors=params["smote_k"], sampling_strategy=strategy, random_state=seed)
    return SMOTE(k_neighbors=params["smote_k"], sampling_strategy=strategy, random_state=seed)

def gen_smote(seed):
    Xr, _ = smote({minority: M}, seed).fit_resample(X_in, y_in)
    return to_raw(pd.DataFrame(Xr, columns=X_in.columns).iloc[len(X_in):]).reset_index(drop=True)

def gen_ctgan(seed, train=None, p=None):
    p = p or params["ctgan"]; set_seed(seed)
    model = CTGAN(epochs=p["epochs"], batch_size=p["batch_size"], pac=p["pac"], verbose=False, enable_gpu=USE_GPU)
    model.fit(real_min if train is None else train, cat_cols)
    return model.sample(n_gen)[X_in.columns]

def gen_tvae(seed):
    p = params["tvae"]; md = SingleTableMetadata(); md.detect_from_dataframe(real_min)
    for c in cat_cols:
        md.update_column(c, sdtype="categorical")
    set_seed(seed)
    model = TVAESynthesizer(metadata=md, epochs=p["epochs"], batch_size=p["batch_size"], embedding_dim=p["embedding_dim"], cuda=USE_GPU)
    model.fit(real_min)
    return model.sample(num_rows=n_gen)[X_in.columns]

def gen_hybrid(seed):
    """Algoritmo 1: SMOTE-NC hasta alpha*M, CTGAN sobre esa minoría ampliada; los interpolados se descartan."""
    Xs, ys = smote({minority: math.ceil(params["alpha_smote"] * M)}, seed).fit_resample(X_in, y_in)
    expanded = to_raw(pd.DataFrame(Xs, columns=X_in.columns)[np.asarray(ys) == minority]).reset_index(drop=True)
    return gen_ctgan(seed, expanded, params["hybrid"])

GENERATORS = {"SMOTE": gen_smote, "CTGAN": gen_ctgan, "TVAE": gen_tvae, "HYBRID": gen_hybrid}

## 4. Fidelidad y utilidad (mismas métricas y clasificadores del artículo)

In [ ]:
fid_sc = StandardScaler().fit(real_min[num_cols])
real_min_s = fid_sc.transform(real_min[num_cols])
REAL_DIV = float(np.mean(pdist(real_min_s[:2000])))

def fidelity(syn):
    s = fid_sc.transform(syn[num_cols].astype(float))
    r = {"Wasserstein": np.mean([wasserstein_distance(real_min_s[:, j], s[:, j]) for j in range(len(num_cols))]),
         "Diversidad": float(np.mean(pdist(pd.DataFrame(s).sample(min(len(s), 2000), random_state=42))))}
    if len(num_cols) > 1:
        rc, sc = real_min[num_cols].corr().fillna(0).values, syn[num_cols].astype(float).corr().fillna(0).values
        r["Dif. correlación"] = float(np.mean(np.abs(rc - sc)[~np.eye(len(num_cols), dtype=bool)]))
    if cat_cols:
        r["Dif. categórica"] = np.mean([sum(abs(real_min[c].value_counts(normalize=True).get(v, 0) -
                                                 syn[c].astype(float).round().value_counts(normalize=True).get(v, 0))
                                             for v in set(real_min[c]) | set(syn[c].astype(float).round())) for c in cat_cols])
    return r

def models(seed, weighted=False):
    w = M / m if weighted else 1
    xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                        eval_metric="logloss", tree_method="hist", random_state=seed, scale_pos_weight=w)
    cat = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, verbose=False, random_seed=seed,
                             class_weights={0: 1, 1: w} if weighted else None)
    return {"XGBoost": xgb, "CatBoost": cat}

def evaluate(model, Xt, yt):
    model.fit(Xt, yt)
    pred, score = model.predict(X_te_s), model.predict_proba(X_te_s)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    rec, spec = tp / (tp + fn), tn / (tn + fp)
    return {"AUC_PR": average_precision_score(y_te, score), "Recall": rec,
            "Precision": precision_score(y_te, pred, zero_division=0), "F1": f1_score(y_te, pred, zero_division=0),
            "GMean": math.sqrt(rec * spec), "BalancedAcc": balanced_accuracy_score(y_te, pred),
            "TP": tp, "FN": fn, "FP": fp, "TN": tn}

## 5. Ejecución: cada corrida regenera los datos sintéticos y reentrena los clasificadores

In [ ]:
util, fid = [], []
t_total = time.time()
for r in range(N_RUNS):
    seed = 42 + r
    sets = {"REAL": (X_in, y_in, False), "REAL_CW": (X_in, y_in, True)}
    for name in METODOS:
        t0 = time.time()
        syn = GENERATORS[name](seed)
        fid.append({"seed": seed, "Método": name, **fidelity(syn), "Tiempo (s)": time.time() - t0})
        sets[name] = (pd.concat([X_in, to_scaled(syn)], ignore_index=True),
                      np.concatenate([y_in, np.full(len(syn), minority)]), False)
    for name, (Xt, yt, w) in sets.items():
        for clf, model in models(seed, w).items():
            util.append({"seed": seed, "Condición": name, "Clasificador": clf, **evaluate(model, Xt, yt)})
    print(f"Corrida {r + 1}/{N_RUNS} (semilla {seed}) lista · {time.time() - t_total:.0f} s acumulados")
util, fid = pd.DataFrame(util), pd.DataFrame(fid)

### Fidelidad (media de las corridas; Wasserstein menor es mejor)

In [ ]:
f = fid.groupby("Método").mean(numeric_only=True).drop(columns="seed").reindex(METODOS)
print(f"Diversidad de los registros minoritarios reales (referencia): {REAL_DIV:.2f}")
f.style.format("{:.3f}").highlight_min(subset=["Wasserstein"], color="#c8e6c9")

### Utilidad (media de las corridas)

In [ ]:
ORDER = ["REAL", "REAL_CW"] + METODOS
METRICS = ["AUC_PR", "Recall", "Precision", "F1", "GMean"]
tab = util.groupby(["Clasificador", "Condición"])[METRICS].mean().reindex(ORDER, level=1)
tab.style.format("{:.3f}").background_gradient(subset=["Recall"], cmap="Greens").background_gradient(subset=["Precision"], cmap="Oranges")

## 6. Validación estadística frente a REAL
Prueba t pareada por semilla y corrección de Holm entre los métodos sintéticos; REAL-CW se contrasta sin corrección. En el artículo, con 10 corridas, se usa Wilcoxon cuando Shapiro-Wilk rechaza la normalidad.

In [ ]:
def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty(len(p)); run = 0
    for k, i in enumerate(order):
        run = max(run, min(1, (len(p) - k) * p[i])); adj[i] = run
    return adj

rows = []
for clf in ["XGBoost", "CatBoost"]:
    base = util[(util.Clasificador == clf) & (util["Condición"] == "REAL")].set_index("seed")
    for met in ["AUC_PR", "Recall", "Precision"]:
        fam = []
        for cond in ["REAL_CW"] + METODOS:
            s = util[(util.Clasificador == clf) & (util["Condición"] == cond)].set_index("seed")
            d = (s[met] - base[met]).values
            p = stats.ttest_rel(s[met], base[met]).pvalue if N_RUNS >= 3 and np.ptp(d) > 1e-9 else np.nan
            fam.append({"Clasificador": clf, "Métrica": met, "Condición": cond, "Δ vs REAL": d.mean(), "p": p})
        syn = [x for x in fam if x["Condición"] != "REAL_CW" and not np.isnan(x["p"])]
        for x, a in zip(syn, holm([x["p"] for x in syn]) if syn else []):
            x["p"] = a
        rows += fam
res = pd.DataFrame(rows)
res["Resultado"] = np.where(res.p.isna(), "sin variación", np.where(res.p < 0.05, np.where(res["Δ vs REAL"] > 0, "▲ sube", "▼ baja"), "sin diferencia"))
res.pivot_table(index=["Clasificador", "Condición"], columns="Métrica", values="Δ vs REAL").reindex(["REAL_CW"] + METODOS, level=1).style.format("{:+.3f}").background_gradient(cmap="RdYlGn", vmin=-0.3, vmax=0.3)

In [ ]:
print(f"Diferencias significativas (p < 0.05) con {N_RUNS} corridas; en el artículo se usaron 10:")
res[res.p < 0.05][["Clasificador", "Métrica", "Condición", "Δ vs REAL", "p", "Resultado"]].round(4)

## 7. Visualización: el desplazamiento de la frontera de decisión

In [ ]:
COLORS = {"REAL": "#555555", "REAL_CW": "#7B5EA7", "SMOTE": "#003264", "CTGAN": "#F3C133", "TVAE": "#8FA6C4", "HYBRID": "#2A9D8F"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
cb = util[util.Clasificador == "CatBoost"].groupby("Condición")[["Recall", "Precision"]].mean().reindex(ORDER)
x = np.arange(len(ORDER))
axes[0].bar(x - 0.2, cb.Recall, 0.4, label="Recall", color="#003264")
axes[0].bar(x + 0.2, cb.Precision, 0.4, label="Precisión", color="#F3C133")
axes[0].set_xticks(x, ORDER, rotation=20); axes[0].set_ylim(0, 1.05); axes[0].legend(); axes[0].set_title("CatBoost: Recall y Precisión")
r_ = np.linspace(0.01, 1, 200)
for f1 in [0.2, 0.4, 0.6, 0.8]:
    p_ = f1 * r_ / (2 * r_ - f1); ok = (p_ > 0) & (p_ <= 1)
    axes[1].plot(r_[ok], p_[ok], ":", color="#C9D1DC"); axes[1].text(1.01, f1 / (2 - f1), f"F1={f1}", fontsize=8, color="grey")
for c in ORDER:
    axes[1].scatter(cb.loc[c, "Recall"], cb.loc[c, "Precision"], s=120, color=COLORS[c], label=c, edgecolor="white", zorder=3)
axes[1].set_xlim(0, 1.1); axes[1].set_ylim(0, 1.05); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precisión")
axes[1].set_title("Puntos de operación (umbral 0.5) y curvas iso-F1"); axes[1].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

## 8. Comparación con los resultados publicados (10 corridas, mismo dataset)

In [ ]:
pub = pd.read_csv(f"{FOLDER}/outputs/metrics_v2/utility_runs.csv")
pub = pub.groupby(["model", "train_set"])[["AUC_PR", "Recall", "Precision"]].mean()
demo = util.groupby(["Clasificador", "Condición"])[["AUC_PR", "Recall", "Precision"]].mean()
pub.index.names = demo.index.names
comp = pub.join(demo, lsuffix=" (artículo)", rsuffix=" (demo)").reindex(ORDER, level=1)
comp[["Recall (artículo)", "Recall (demo)", "AUC_PR (artículo)", "AUC_PR (demo)"]].style.format("{:.3f}")

**Lectura.** La síntesis desplaza la frontera de decisión hacia la clase minoritaria: el Recall sube y la Precisión baja, y los puntos se mueven a lo largo de las curvas iso-F1 más que hacia arriba. Los pesos de clase (REAL-CW) logran un desplazamiento similar sin generar registros. Las diferencias con el artículo se deben al menor número de corridas y al hardware (GPU frente a CPU).